In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import shutil
import math
import os
import json
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm, trange
import torchmetrics
import matplotlib.pyplot as plt
from pprint import pprint
import utils as course_utils

# ======== Your controls: change these to run the notebook your way ========
# True: a small run on a tiny generated dataset; grading always uses it.
# False: training on real MNIST (downloaded to ~/ once, about 60 MB).
IS_UNITTEST = True
# Epochs for each of the six networks in the experiment when IS_UNITTEST =
# False (about 10 minutes in total on a laptop CPU for 3 epochs).
EXPERIMENT_EPOCHS = 3
# ==========================================================================
# Grading always uses the small run, whatever IS_UNITTEST is set to above.
IS_UNITTEST = IS_UNITTEST or course_utils.checker.is_unittest_run()
# Thread count changes the order of float sums; the small run must
# repeat the teacher's numbers on any machine.
if IS_UNITTEST:
    torch.set_num_threads(1)

# -- start --
data_dir = course_utils.checker.get_data_dir()
course_utils.checker.save_npz({'seed': np.array(0)}, data_dir / 'inputs_public.npz')
course_utils.checker.save_npz({'seed': np.array(42)}, data_dir / 'inputs_private.npz')
if not course_utils.checker.is_valid_npz('xinputs.npz'):
    shutil.copyfile(data_dir / 'inputs_public.npz', 'xinputs.npz')
# -- end --

seed = int(course_utils.checker.load_answer('xinputs.npz')['seed'])

import source
# -- start --
# The teacher's notebook runs with the teacher's copy of the shared code.
if course_utils.hidden.detect():
    course_utils.hidden.replace_code()
# -- end --
src = source.code.src
answer = {}


# Deep Networks, Batch Normalization and Residual Blocks

A deeper network can describe more complicated functions, but simply
stacking layers makes it hard to train: the signal and its gradient fade
or blow up on their way through dozens of layers. In this notebook we
build deep fully connected networks for MNIST, look at their signals and
gradients layer by layer, and train six versions side by side: shallow,
deep, deep with batch normalization (before or after the linear layers),
deep with residual connections, and both.

We reuse the dataset, the loss, the metric and the training loop from the
Shallow Feedforward NNs task.

**Files to edit:**
- [src/models/feedforward/deep_batchnorm_net.py](../../source/code/src/models/feedforward/deep_batchnorm_net.py): the network, the bottleneck block, batch normalization and the residual block;
- the marked cells of this notebook: the six networks, the loss and the metric.


In [ ]:
src.utils.seed.seed_all(0)


# The Data

The same `MNISTSimpleDataset` as in the Shallow Feedforward NNs task: in
the small run, a tiny generated stand-in; in a full run, real MNIST.


In [ ]:
MNIST_train = src.datasets.mnist_simple.MNISTSimpleDataset(train=True, small=IS_UNITTEST, seed=seed)
MNIST_valid = src.datasets.mnist_simple.MNISTSimpleDataset(train=False, small=IS_UNITTEST, seed=seed)
print('training images:', len(MNIST_train), 'validation images:', len(MNIST_valid))

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for index, ax in enumerate(axes):
    sample = MNIST_train[index]
    ax.imshow(sample['image'], cmap='gray')
    ax.set_title(f"label {int(sample['label'])}")
    ax.axis('off')
plt.show()

# A fixed batch for the checks of the next tasks.
check_generator = torch.Generator().manual_seed(0)
check_images = torch.randn(10, 28, 28, generator=check_generator)


def run_check(network):
    """Predictable weights, then one pass over the fixed check batch."""
    src.utils.deterministic_init(network)
    batch = network({'data': {'image': check_images.clone()}})
    return {'output': batch['signals']['output'], 'class': batch['postprocessed']['class']}


# Task 1: Build the Deep Network

## Goal

Implement `DeepFullyConnectedNet` in [src/models/feedforward/deep_batchnorm_net.py](../../source/code/src/models/feedforward/deep_batchnorm_net.py).

The network:
- flattens the image to a vector of `dim_input` numbers;
- projects it to `dim_embed` numbers with an **encoder**, one `torch.nn.Linear`;
- passes it through `n_blocks` **blocks**, one after another; every block
  keeps the size `dim_embed`;
- turns the result into `dim_output` logits with a **decoder**, one `torch.nn.Linear`.

`block` is a function: `block(dim_embed)` makes one new block. By default a
block is a single `torch.nn.Linear(dim_embed, dim_embed)`; in the next tasks
we pass better blocks.

## Parts To Implement

- `__init__`: create `self.encoder`, `self.blocks` (a `torch.nn.Sequential`
  of `n_blocks` blocks made by `block(dim_embed)`), and `self.decoder`. They
  replace the placeholder layers that are already there.
- `postprocessing`: turn the logits in `batch['signals']['output']` into
  predicted classes with `argmax`.

The rest is given. `__forward_kernel` flattens the images and passes them
through the encoder, the blocks, and the decoder; `forward` follows the
batch protocol of the Shallow Feedforward NNs task: it reads
`batch['data']['image']` and writes `batch['signals']['output']` and
`batch['postprocessed']['class']`.


In [ ]:
# Edit: ../../source/code/src/models/feedforward/deep_batchnorm_net.py
simple_network = src.models.feedforward.deep_batchnorm_net.DeepFullyConnectedNet(
    dim_input=28 * 28, dim_embed=64, dim_output=10, n_blocks=3)
# ^^^ This class should be implemented in this task

print(simple_network)
answer['simple_network'] = run_check(simple_network)
print('logits:', tuple(answer['simple_network']['output'].shape))


In [ ]:
course_utils.checker.print_progress(
    'Task 1: Build the Deep Network', answer,
    prefixes=('simple_network',),
    is_small_run=IS_UNITTEST)


# Task 2: The Bottleneck Block

## Goal

Implement `Bottleneck` in [src/models/feedforward/deep_batchnorm_net.py](../../source/code/src/models/feedforward/deep_batchnorm_net.py):
two linear layers that first shrink the signal by `compression` and then
restore its size, with an activation in between:

- `prenormalization()`;
- `Linear(in_channels, in_channels // compression)`;
- `postnormalization()`;
- `activation()`;
- `prenormalization()`;
- `Linear(in_channels // compression, in_channels)`;
- `postnormalization()`.

`prenormalization`, `postnormalization` and `activation` are passed as
classes (constructors): `torch.nn.Identity` by default, so without them a
block is just `Linear → ReLU → Linear`. In Task 5 we pass `BatchNorm` as
one of them. Put the layers into `self.block`, a `torch.nn.Sequential`.

The check builds a network of 3 bottleneck blocks with `compression=4`.


In [ ]:
# Edit: ../../source/code/src/models/feedforward/deep_batchnorm_net.py
bottleneck_network = src.models.feedforward.deep_batchnorm_net.DeepFullyConnectedNet(
    dim_input=28 * 28, dim_embed=64, dim_output=10, n_blocks=3,
    block=lambda n_channels: src.models.feedforward.deep_batchnorm_net.Bottleneck(n_channels, compression=4))
# ^^^ This class should be implemented in this task

print(bottleneck_network)
answer['bottleneck_network'] = run_check(bottleneck_network)


In [ ]:
course_utils.checker.print_progress(
    'Task 2: The Bottleneck Block', answer,
    prefixes=('bottleneck_network',),
    is_small_run=IS_UNITTEST)


# Task 3: Batch Normalization

## Goal

Implement `BatchNorm.forward` in [src/models/feedforward/deep_batchnorm_net.py](../../source/code/src/models/feedforward/deep_batchnorm_net.py).
Batch normalization gives every channel $k$ of the signal mean 0 and
standard deviation 1:

$$X^*_{\cdot k} = \frac{X_{\cdot k} - \mu_k}{\sigma_k + \varepsilon}.$$

- **Training mode** (`self.training`): $\mu_k$ and $\sigma_k^2$ are the mean
  and the **biased** variance (the mean of squared deviations) of channel
  $k$ over the batch. Also update the running averages, in place and
  without gradients:
  $$\bar\mu \leftarrow \beta \bar\mu + (1 - \beta)\,\mu, \qquad \overline{\sigma^2} \leftarrow \beta \overline{\sigma^2} + (1 - \beta)\,\sigma^2.$$
- **Evaluation mode**: a single example has no batch to compute statistics
  from, so use the running averages: $\mu = \bar\mu$, $\sigma = \sqrt{\overline{\sigma^2}}$.

`self.running_mean` and `self.running_var` are created for you with the
right shape (`[1, channels]`), starting at 0 and 1.

The check normalizes three batches in training mode, then one in evaluation mode.


In [ ]:
# Edit: ../../source/code/src/models/feedforward/deep_batchnorm_net.py
batch_norm = src.models.feedforward.deep_batchnorm_net.BatchNorm(beta=0.9)
# ^^^ This class should be implemented in this task

norm_generator = torch.Generator().manual_seed(1)
norm_batches = [torch.randn(16, 4, generator=norm_generator) * 3 + 2 for _ in range(4)]
training_outputs = [batch_norm(batch) for batch in norm_batches[:3]]
batch_norm.eval()
evaluation_output = batch_norm(norm_batches[3])

print('training output, mean and std per channel:',
      training_outputs[0].mean(dim=0), training_outputs[0].std(dim=0, unbiased=False))
answer['batch_norm'] = {
    'training_output': training_outputs[0],
    'running_mean': batch_norm.running_mean,
    'running_var': batch_norm.running_var,
    'evaluation_output': evaluation_output,
}


In [ ]:
course_utils.checker.print_progress(
    'Task 3: Batch Normalization', answer,
    prefixes=('batch_norm',),
    is_small_run=IS_UNITTEST)


# Task 4: The Residual Block

## Goal

Implement `Residual` in [src/models/feedforward/deep_batchnorm_net.py](../../source/code/src/models/feedforward/deep_batchnorm_net.py).
It wraps another module $f$ and adds its input to its output:

$$\mathrm{Residual}(x) = x + f(x).$$

Even when $f$ passes almost nothing through, the signal and its gradient
still travel along the $x$ term, however deep the network is.


In [ ]:
# Edit: ../../source/code/src/models/feedforward/deep_batchnorm_net.py
residual_network = src.models.feedforward.deep_batchnorm_net.DeepFullyConnectedNet(
    dim_input=28 * 28, dim_embed=64, dim_output=10, n_blocks=3,
    block=lambda n_channels: src.models.feedforward.deep_batchnorm_net.Residual(src.models.feedforward.deep_batchnorm_net.Bottleneck(n_channels)))
# ^^^ This class should be implemented in this task

answer['residual_network'] = run_check(residual_network)


In [ ]:
course_utils.checker.print_progress(
    'Task 4: The Residual Block', answer,
    prefixes=('residual_network',),
    is_small_run=IS_UNITTEST)


# Task 5: Six Networks

## Goal

`make_network(block, n_blocks)` builds a `DeepFullyConnectedNet` for MNIST
with the given block constructor. Fill in `network_blocks`: for every name,
the block constructor and the number of blocks.

| name | blocks |
|---|---|
| `shallow` | 1 plain `Bottleneck` |
| `deep` | 50 plain `Bottleneck` blocks |
| `deep_prenorm` | 50 `Bottleneck` blocks with `prenormalization=BatchNorm` |
| `deep_postnorm` | 50 `Bottleneck` blocks with `postnormalization=BatchNorm` |
| `deep_residual` | 50 `Residual(Bottleneck)` blocks |
| `deep_residual_prenorm` | 50 `Residual(Bottleneck)` blocks with `prenormalization=BatchNorm` |

A block constructor takes the number of channels: for example
`lambda n_channels: Bottleneck(n_channels, prenormalization=BatchNorm)`.


In [ ]:
# Edit this cell in task.ipynb.
Bottleneck = src.models.feedforward.deep_batchnorm_net.Bottleneck
BatchNorm = src.models.feedforward.deep_batchnorm_net.BatchNorm
Residual = src.models.feedforward.deep_batchnorm_net.Residual


def make_network(block, n_blocks):
    return src.models.feedforward.deep_batchnorm_net.DeepFullyConnectedNet(
        dim_input=28 * 28, dim_embed=64, dim_output=10, block=block, n_blocks=n_blocks)


network_blocks = {name: (Bottleneck, 1) for name in [
    'shallow', 'deep', 'deep_prenorm', 'deep_postnorm', 'deep_residual', 'deep_residual_prenorm']}

## YOUR CODE HERE
# -- start --
network_blocks = {
    'shallow': (Bottleneck, 1),
    'deep': (Bottleneck, 50),
    'deep_prenorm': (lambda n_channels: Bottleneck(n_channels, prenormalization=BatchNorm), 50),
    'deep_postnorm': (lambda n_channels: Bottleneck(n_channels, postnormalization=BatchNorm), 50),
    'deep_residual': (lambda n_channels: Residual(Bottleneck(n_channels)), 50),
    'deep_residual_prenorm': (
        lambda n_channels: Residual(Bottleneck(n_channels, prenormalization=BatchNorm)), 50),
}
# -- end --

answer['networks'] = {}
for name, (block, n_blocks) in network_blocks.items():
    network = make_network(block, n_blocks)
    answer['networks'][name] = {
        'parameter_count': sum(p.numel() for p in network.parameters()),
        **run_check(network),
    }
    print(f"{name:22s} {n_blocks:3d} blocks, "
          f"{answer['networks'][name]['parameter_count']} parameters")


In [ ]:
course_utils.checker.print_progress(
    'Task 5: Six Networks', answer,
    prefixes=('networks',),
    is_small_run=IS_UNITTEST)


## The Network Graph

`DeepFullyConnectedNet` has three parts: the `encoder`, the `blocks` and the
`decoder`. The graph below is drawn from the tensor operations of
`__forward_kernel` for a small network of 3 residual blocks with
prenormalization; it needs the Graphviz `dot` program and is skipped in the
small run.


In [ ]:
graph_network = make_network(network_blocks['deep_residual_prenorm'][0], n_blocks=3)
print(graph_network)
if not IS_UNITTEST and shutil.which('dot'):
    model_graph = src.utils.model_visualization.draw_forward_kernel_graph(graph_network, depth=3)
    display(model_graph.visual_graph)
elif not IS_UNITTEST:
    print('Install the Graphviz dot executable to display this graph.')


# Task 6: Create the Loss and the Metric

## Goal

The same loss and metric as in the Shallow Feedforward NNs task, following
the batch protocol:

- `loss`: the cross-entropy of the logits `batch['signals']['output']` and
  the labels `batch['data']['label']`:
  $L = -\frac{1}{N} \sum_i \log \mathrm{softmax}(z_i)_{y_i}$;
- `accuracy`: the share of images whose predicted class
  `batch['postprocessed']['class']` equals the label. Return the numerator
  (the number of correct predictions) and the denominator (the batch
  size), so that `train_model` can sum them over all batches.


In [ ]:
# Edit this cell in task.ipynb.
def loss(batch):
    return batch['signals']['output'].abs().sum()


def accuracy(batch):
    return 0, 1

## YOUR CODE HERE
# -- start --
def loss(batch):
    return torch.nn.functional.cross_entropy(batch['signals']['output'], batch['data']['label'])


def accuracy(batch):
    correct = (batch['postprocessed']['class'] == batch['data']['label']).float().sum()
    return correct, len(batch['data']['label'])
# -- end --

losses = {'ce': loss}
metrics = {'accuracy': accuracy}

# Fixed logits for three images: the first two are classified right.
scored_batch = {
    'signals': {'output': torch.tensor([[2.0, 0.0, -1.0], [0.0, 1.0, 0.5], [3.0, 0.0, 0.0]])},
    'postprocessed': {'class': torch.tensor([0, 1, 0])},
    'data': {'label': torch.tensor([0, 1, 2])},
}
answer['test_loss_val'] = {name: fn(scored_batch) for name, fn in losses.items()}
answer['test_metrics_val'] = {name: fn(scored_batch) for name, fn in metrics.items()}
pprint(answer['test_loss_val'])
pprint(answer['test_metrics_val'])


In [ ]:
course_utils.checker.print_progress(
    'Task 6: Create the Loss and the Metric', answer,
    prefixes=('test_loss_val', 'test_metrics_val'),
    is_small_run=IS_UNITTEST)


# Signals and Gradients, Layer by Layer

The data loaders are the ones from the Shallow Feedforward NNs task. For
each of the six networks, fresh and untrained, we pass one batch forward,
compute the loss, and backpropagate. The plots show, for every layer, the
size of the weights and of the gradients that reach them.


In [ ]:
batch_size = 32 if IS_UNITTEST else 128
train_dl = torch.utils.data.DataLoader(
    MNIST_train, batch_size=batch_size, shuffle=True, drop_last=True,
    generator=torch.Generator().manual_seed(0))
valid_dl = torch.utils.data.DataLoader(MNIST_valid, batch_size=batch_size, shuffle=False)
gradient_batch = next(iter(valid_dl))

for name, (block, n_blocks) in network_blocks.items():
    src.utils.seed.seed_all(0)
    network = make_network(block, n_blocks)
    network.train()
    batch = network({'data': gradient_batch})
    losses['ce'](batch).backward()
    print(f'{name}: {n_blocks} block(s)')
    src.utils.model_visualization.plot_signal_stats(network, {'data': gradient_batch})


What to look for:

- **deep, plain**: the gradients shrink by about ten orders of magnitude
  from the decoder to the first blocks, so the early layers do not learn
  at all (vanishing gradients);
- **deep, with batch normalization** (before or after the linear layers):
  the signal of every layer stays at the same scale, but the gradients now
  *grow* by about six orders of magnitude towards the first blocks
  (exploding gradients). Each normalization divides by a standard
  deviation below 1, and backpropagation multiplies by the same factor in
  every block;
- **deep, residual**: the gradient reaches every block through the $x$
  terms, so all blocks get gradients of a similar size;
- **deep, residual with batch normalization**: still nearly even, growing
  only a little towards the input.

Residual connections keep the gradient even *across* layers; batch
normalization keeps the signal even *within* a layer.


# Training the Six Networks

Each network is trained by `src.train_loop.train_model`, the loop from the
Shallow Feedforward NNs task, with `AdamW` (`lr=1e-3`): one epoch in the
small run, `EXPERIMENT_EPOCHS` epochs in a full run.


In [ ]:
n_epochs = 1 if IS_UNITTEST else EXPERIMENT_EPOCHS
histories = {}
for name, (block, n_blocks) in network_blocks.items():
    src.utils.seed.seed_all(0)
    network = make_network(block, n_blocks)
    optimizer = torch.optim.AdamW(network.parameters(), lr=1.0e-3)
    history = src.train_loop.train_model(
        network, n_epochs, train_dl, valid_dl, losses, optimizer,
        metrics=metrics, run_name=name)
    histories[name] = {
        'train_loss': history['train_loss']['ce'],
        'valid_loss': history['valid_loss']['ce'],
        'valid_accuracy': history['valid_metrics']['accuracy'],
    }

answer['experiments'] = histories


In [ ]:
course_utils.checker.print_progress(
    'Training the Six Networks', answer,
    prefixes=('experiments',),
    is_small_run=IS_UNITTEST)


# What the Experiment Shows

The validation accuracy of the six networks after every epoch. In a full
run, look at which deep networks learn at all, and which train as fast as
the shallow one or faster.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for name, history in histories.items():
    epochs = range(1, len(history['valid_loss']) + 1)
    axes[0].plot(epochs, history['valid_loss'], 'o-', label=name)
    axes[1].plot(epochs, history['valid_accuracy'], 'o-', label=name)
axes[0].set_title('validation loss')
axes[1].set_title('validation accuracy')
for ax in axes:
    ax.set_xlabel('epoch')
    ax.legend()
plt.show()

for name, history in histories.items():
    print(f"{name:22s} validation accuracy {history['valid_accuracy'][-1]:.3f}")


In [ ]:
course_utils.checker.save_answer(answer)
